# SIMBES — Notebook M2: Diseño Hidráulico

**Objetivo pedagógico**: comprender cómo se calculan las pérdidas de carga en un sistema BES,
cómo se selecciona el número de etapas, y cómo la curva de sistema intersecta la curva H-Q de la bomba.

**Física cubierta**:
- TDH = H_estático + H_fricción (Darcy-Weisbach + Colebrook-White) + H_contrapresión
- Velocidad específica (Ns) y tipo de impulsor
- Cálculo de número de etapas
- Curva de sistema vs. curva H-Q multietapa

---

In [ ]:
import sys
sys.path.insert(0, '../backend')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

from physics.hydraulics import (
    reynolds_number, colebrook_white, flow_regime,
    tdh_components, pump_head_total_m,
    compute_required_stages, find_hydraulic_op_point,
    pump_specific_speed, affinity_laws,
    STB_TO_M3, FT_TO_M,
)

plt.style.use('dark_background')
print('✓ Módulos cargados correctamente')

## 1. Parámetros base del sistema

In [ ]:
# ── Parámetros del pozo ──────────────────────────────────────────
depth_m     = 1800.0   # m    — Profundidad de la bomba
Pwh_psi     = 150.0    # psi  — Presión de cabezal
rho_kgL     = 0.876    # kg/L — Densidad del fluido
mu_cP       = 3.0      # cP   — Viscosidad dinámica

# ── Tubing ───────────────────────────────────────────────────────
D_in        = 2.441    # pulgadas (2-7/8" OD, ID = 2.441")

# ── Bomba BES ────────────────────────────────────────────────────
H0_stage_ft = 45.0     # ft/etapa — cabeza por etapa a Q=0, 60 Hz
freq_hz     = 60.0     # Hz

# ── Caudal de diseño (BEP a 60 Hz) ──────────────────────────────
Q_bep_stbd = 2100 * (freq_hz / 60)    # STB/d
Q_bep_m3d  = Q_bep_stbd * STB_TO_M3   # m³/d

# ── Resultados en BEP ────────────────────────────────────────────
comps = tdh_components(Q_bep_m3d, depth_m, Pwh_psi, D_in, mu_cP, rho_kgL)
n_stages = compute_required_stages(depth_m, Pwh_psi, D_in, mu_cP, rho_kgL, freq_hz, H0_stage_ft)
ns_info  = pump_specific_speed(H0_stage_ft)

print('──── TDH en BEP ────────────────────────────────────────────')
print(f'  H estático      = {comps["H_static_m"]:>8.1f} m   ({depth_m:.0f} m de columna de fluido)')
print(f'  H fricción      = {comps["H_friction_m"]:>8.1f} m   ({comps["H_friction_m"]/comps["TDH_m"]*100:.1f}% del TDH)')
print(f'  H contrapresión = {comps["H_back_m"]:>8.1f} m   ({Pwh_psi:.0f} psi cabezal)')
print(f'  TDH TOTAL       = {comps["TDH_m"]:>8.1f} m')
print()
print('──── Hidráulica del fluido ─────────────────────────────────')
print(f'  Velocidad       = {comps["v_ms"]:>8.3f} m/s')
print(f'  Reynolds        = {comps["Re"]:>8,.0f}   ({comps["regime"]})')
print(f'  Factor f (C-W)  = {comps["f"]:>8.5f}')
print()
print('──── Bomba BES ─────────────────────────────────────────────')
print(f'  Etapas requeridas = {n_stages}')
print(f'  Ns               = {ns_info["Ns"]:,}  →  {ns_info["type"]}')

## 2. Colebrook-White: factor de fricción vs. Reynolds

La ecuación de Colebrook-White es implícita en f y se resuelve iterativamente:

$$\frac{1}{\sqrt{f}} = -2 \log_{10}\!\left(\frac{\varepsilon}{3.7 D} + \frac{2.51}{Re\sqrt{f}}\right)$$

Para flujo laminar: $f = 64/Re$ (Hagen-Poiseuille)

In [ ]:
Re_arr  = np.logspace(2, 7, 500)
D_m     = D_in * 0.0254
f_arr   = np.array([colebrook_white(Re, D_m) for Re in Re_arr])

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_facecolor('#0D1424')
fig.patch.set_facecolor('#0B0F1A')

# Región laminar
ax.axvspan(100, 2300, alpha=0.12, color='#818CF8', label='Laminar (f = 64/Re)')
ax.axvspan(2300, 4000, alpha=0.08, color='#F59E0B', label='Transición')
ax.axvspan(4000, 1e7, alpha=0.06, color='#34D399', label='Turbulento (Colebrook-White)')

ax.loglog(Re_arr, f_arr, color='#FB923C', lw=2.5, label='Factor f (Colebrook-White)')

# Punto de operación actual
Re_op = comps['Re']
f_op  = comps['f']
ax.scatter(Re_op, f_op, color='#FB7185', s=100, zorder=10, label=f'Operación: Re={Re_op:,.0f}, f={f_op:.4f}')

ax.set_xlabel('Número de Reynolds (Re)', color='#94A3B8', fontsize=11)
ax.set_ylabel('Factor de fricción de Darcy (f)', color='#94A3B8', fontsize=11)
ax.set_title('Diagrama de Moody — Colebrook-White', color='#E2E8F0', fontsize=13, pad=12)
ax.tick_params(colors='#64748B')
for spine in ax.spines.values():
    spine.set_color('#1E293B')
ax.grid(color='#1E293B', lw=0.6, which='both')
ax.legend(facecolor='#111827', edgecolor='#1E293B', labelcolor='#CBD5E1', fontsize=10)
ax.set_xlim(100, 1e7)
ax.set_ylim(0.005, 0.15)
plt.tight_layout()
plt.show()

## 3. Curva de sistema vs. H-Q de la bomba multietapa

- **Curva de sistema**: TDH(Q) = H_estático + H_fricción(Q) + H_contrapresión. Crece con Q² (fricción).
- **Curva H-Q bomba**: H_total(Q) = H₀ × N_etapas × (1 − (Q_ref/Q_max)^1.85) × (f/60)². Cae con Q.
- **Punto de operación**: intersección de ambas curvas.

In [ ]:
def build_curves(depth_m, Pwh_psi, D_in, mu_cP, rho_kgL, freq_hz, H0_stage_ft, N_pts=150):
    """Construye curvas de sistema y H-Q bomba."""
    n_stages = compute_required_stages(depth_m, Pwh_psi, D_in, mu_cP, rho_kgL, freq_hz, H0_stage_ft)
    Qmax_m3d = 4200 * (freq_hz / 60) * STB_TO_M3 * 1.15
    Q_arr    = np.linspace(0, Qmax_m3d, N_pts)
    
    tdh_arr  = np.array([tdh_components(q, depth_m, Pwh_psi, D_in, mu_cP, rho_kgL)['TDH_m'] for q in Q_arr])
    pump_arr = np.array([pump_head_total_m(q, freq_hz, n_stages, H0_stage_ft) for q in Q_arr])
    
    op = find_hydraulic_op_point(depth_m, Pwh_psi, D_in, mu_cP, rho_kgL, freq_hz, n_stages, H0_stage_ft)
    Q_bep = 2100 * (freq_hz / 60) * STB_TO_M3
    
    return Q_arr, tdh_arr, pump_arr, op, n_stages, Q_bep


def plot_system(depth_m, Pwh_psi, D_in, mu_cP, rho_kgL, freq_hz, H0_stage_ft):
    Q_arr, tdh_arr, pump_arr, op, n_stages, Q_bep = build_curves(
        depth_m, Pwh_psi, D_in, mu_cP, rho_kgL, freq_hz, H0_stage_ft
    )
    comps    = tdh_components(Q_bep, depth_m, Pwh_psi, D_in, mu_cP, rho_kgL)
    ns_info  = pump_specific_speed(H0_stage_ft)
    H_static = depth_m
    H_back   = comps['H_back_m']

    fig, ax = plt.subplots(figsize=(13, 7))
    ax.set_facecolor('#0D1424')
    fig.patch.set_facecolor('#0B0F1A')

    # Curva de sistema
    ax.plot(Q_arr, tdh_arr, color='#FB923C', lw=2.5, label='Curva de sistema (TDH)')
    # Curva H-Q bomba
    ax.plot(Q_arr, pump_arr, color='#38BDF8', lw=2.5, label=f'H-Q Bomba ({n_stages} etapas, {freq_hz:.0f} Hz)')

    # Componentes TDH (líneas horizontales de referencia)
    ax.axhline(H_static, color='#64748B', lw=1, linestyle='--', alpha=0.7)
    ax.axhline(H_static + H_back, color='#FBBF24', lw=1, linestyle='--', alpha=0.7)
    ax.text(Q_arr[-1]*0.02, H_static + 8, f'H_estático = {H_static:.0f} m', color='#64748B', fontsize=9)
    ax.text(Q_arr[-1]*0.02, H_static + H_back + 8, f'+ H_back = {H_static+H_back:.0f} m', color='#FBBF24', fontsize=9)

    # BEP
    ax.axvline(Q_bep, color='#F472B6', lw=1.2, linestyle=':', alpha=0.8)
    ax.text(Q_bep + 1, pump_arr.max() * 0.95, f'BEP\n{Q_bep:.0f} m³/d', color='#F472B6', fontsize=9, ha='left')

    # Punto de operación
    if op:
        ax.scatter(op['Q_m3d'], op['TDH_m'], color='#FB7185', s=120, zorder=10, label='Punto de Operación')
        ax.annotate(
            f"  Q = {op['Q_m3d']:.0f} m³/d\n  TDH = {op['TDH_m']:.0f} m",
            xy=(op['Q_m3d'], op['TDH_m']),
            xytext=(op['Q_m3d'] + Q_arr[-1]*0.05, op['TDH_m'] + 30),
            color='#FB7185', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#FB7185', lw=1)
        )

    ax.set_xlabel('Caudal Q (m³/d)', color='#94A3B8', fontsize=11)
    ax.set_ylabel('Cabeza (m)', color='#94A3B8', fontsize=11)
    ax.set_title(
        f'Diseño Hidráulico BES — depth={depth_m:.0f}m · D={D_in}" · {freq_hz:.0f}Hz · Ns={ns_info["Ns"]:,} ({ns_info["type"]})',
        color='#E2E8F0', fontsize=12, pad=12
    )
    ax.tick_params(colors='#64748B')
    for spine in ax.spines.values():
        spine.set_color('#1E293B')
    ax.grid(color='#1E293B', lw=0.7)
    ax.legend(facecolor='#111827', edgecolor='#1E293B', labelcolor='#CBD5E1', fontsize=10)
    ax.set_xlim(0, Q_arr[-1])
    ax.set_ylim(0, max(tdh_arr.max(), pump_arr.max()) * 1.1)
    plt.tight_layout()
    plt.show()
    print(f'\nEtapas requeridas: {n_stages}')
    print(f'TDH en BEP:  {comps["TDH_m"]:.1f} m  (H_fric = {comps["H_friction_m"]:.1f} m = {comps["H_friction_m"]/comps["TDH_m"]*100:.1f}% del TDH)')
    if op:
        print(f'Q operativo: {op["Q_m3d"]:.1f} m³/d  |  TDH operativo: {op["TDH_m"]:.1f} m')


plot_system(depth_m, Pwh_psi, D_in, mu_cP, rho_kgL, freq_hz, H0_stage_ft)

## 4. Exploración Interactiva

In [ ]:
@interact(
    depth_m    = IntSlider(value=1800, min=300,  max=4300, step=100, description='Profundidad (m)'),
    Pwh_psi    = IntSlider(value=150,  min=50,   max=1000, step=25,  description='Pwh (psi)'),
    D_in       = Dropdown(options=[1.995, 2.441, 2.992, 3.958], value=2.441, description='D_in (pulg)'),
    mu_cP      = FloatSlider(value=3.0, min=0.5, max=50.0, step=0.5, description='Visc. (cP)'),
    freq_hz    = FloatSlider(value=60.0, min=30.0, max=70.0, step=1.0, description='Freq. (Hz)'),
    H0_stage_ft= IntSlider(value=45, min=15, max=100, step=1, description='H₀/etapa (ft)'),
)
def interactive_hydraulics(depth_m, Pwh_psi, D_in, mu_cP, freq_hz, H0_stage_ft):
    plot_system(depth_m, Pwh_psi, D_in, mu_cP, 0.876, freq_hz, H0_stage_ft)

## 5. Sensibilidad — Impacto del diámetro de tubing en fricción

Las pérdidas por fricción escalan como $h_f \propto 1/D^5$ (Darcy-Weisbach con $v \propto 1/D^2$).

In [ ]:
tubing_sizes = [
    ('2⅜" (1.995")',  1.995),
    ('2⅞" (2.441")',  2.441),
    ('3½" (2.992")',  2.992),
    ('4½" (3.958")',  3.958),
]

Q_base = Q_bep_m3d   # Usar BEP como caudal de análisis

results = []
for label, d in tubing_sizes:
    c = tdh_components(Q_base, depth_m, Pwh_psi, d, mu_cP, rho_kgL)
    n = compute_required_stages(depth_m, Pwh_psi, d, mu_cP, rho_kgL, freq_hz, H0_stage_ft)
    results.append((label, d, c['H_friction_m'], c['TDH_m'], c['Re'], c['regime'], n))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0B0F1A')
colors_bar = ['#818CF8', '#34D399', '#38BDF8', '#FB923C']

labels  = [r[0] for r in results]
h_frict = [r[2] for r in results]
n_stage = [r[6] for r in results]

for ax in axes:
    ax.set_facecolor('#0D1424')
    ax.tick_params(colors='#64748B')
    for spine in ax.spines.values():
        spine.set_color('#1E293B')
    ax.grid(color='#1E293B', lw=0.7, axis='y')

bars1 = axes[0].bar(labels, h_frict, color=colors_bar, alpha=0.85)
axes[0].set_ylabel('H fricción (m)', color='#94A3B8')
axes[0].set_title('Pérdidas por fricción vs. Tubing', color='#E2E8F0')
axes[0].set_xticklabels(labels, color='#94A3B8', fontsize=9)
for bar, val in zip(bars1, h_frict):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{val:.0f} m', ha='center', color='#CBD5E1', fontsize=9)

bars2 = axes[1].bar(labels, n_stage, color=colors_bar, alpha=0.85)
axes[1].set_ylabel('Número de etapas', color='#94A3B8')
axes[1].set_title('Etapas requeridas vs. Tubing', color='#E2E8F0')
axes[1].set_xticklabels(labels, color='#94A3B8', fontsize=9)
for bar, val in zip(bars2, n_stage):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(val), ha='center', color='#CBD5E1', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n{"-"*80}')
print(f'  {"Tubing":<22} {"D_in":>6}  {"H_fric (m)":>12}  {"Re":>10}  {"Régimen":<12}  {"Etapas":>8}')
print(f'{"-"*80}')
for label, d, hf, tdh, re, reg, ns in results:
    print(f'  {label:<22} {d:>6.3f}  {hf:>12.1f}  {re:>10,.0f}  {reg:<12}  {ns:>8}')
print(f'{"-"*80}')

## 6. Velocidad Específica (Ns) y tipo de impulsor

$$N_s = \frac{N \cdot \sqrt{Q_{BEP}}}{H_{BEP}^{0.75}} \quad \text{(unidades US: rpm, GPM, ft)}$$

| Rango Ns | Tipo | Características |
|---|---|---|
| < 1 500 | Radial | Alta cabeza/etapa, bajo caudal |
| 1 500–4 500 | Flujo Mixto | Mayoría de BES modernos |
| > 4 500 | Axial | Alto caudal, baja cabeza/etapa |

In [ ]:
H0_range = np.linspace(15, 100, 200)
Ns_arr   = np.array([pump_specific_speed(h)['Ns'] for h in H0_range])

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_facecolor('#0D1424')
fig.patch.set_facecolor('#0B0F1A')

ax.axhspan(0,    1500, alpha=0.15, color='#818CF8', label='Radial (Ns < 1 500)')
ax.axhspan(1500, 4500, alpha=0.12, color='#34D399', label='Flujo Mixto (1 500–4 500) ← BES típico')
ax.axhspan(4500, 6000, alpha=0.10, color='#F59E0B', label='Axial (Ns > 4 500)')

ax.plot(H0_range, Ns_arr, color='white', lw=2.5)

# Punto actual
Ns_current = pump_specific_speed(H0_stage_ft)['Ns']
ax.scatter(H0_stage_ft, Ns_current, color='#FB7185', s=120, zorder=10, label=f'H₀={H0_stage_ft} ft → Ns={Ns_current:,}')
ax.axhline(1500, color='#818CF8', lw=1, linestyle='--', alpha=0.5)
ax.axhline(4500, color='#F59E0B', lw=1, linestyle='--', alpha=0.5)

ax.set_xlabel('H₀ por etapa a Q=0 (ft) — a 60 Hz', color='#94A3B8', fontsize=11)
ax.set_ylabel('Velocidad Específica Ns (US units)', color='#94A3B8', fontsize=11)
ax.set_title('Ns vs. Cabeza por Etapa — Clasificación del Impulsor BES', color='#E2E8F0', fontsize=12, pad=12)
ax.tick_params(colors='#64748B')
for spine in ax.spines.values():
    spine.set_color('#1E293B')
ax.grid(color='#1E293B', lw=0.7)
ax.legend(facecolor='#111827', edgecolor='#1E293B', labelcolor='#CBD5E1', fontsize=10)
ax.set_xlim(15, 100)
ax.set_ylim(0, 6000)
plt.tight_layout()
plt.show()